In [2]:
import os
import json
import re
import xml.etree.ElementTree as ET

import pandas as pd
import numpy as np

import networkx as nx

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [3]:
def detect_stix_version(file_path):
    if file_path.endswith(".xml"):
        return "1.x"
    
    with open(file_path, "r") as f:
        data = json.load(f)
    
    if "spec_version" in data:
        return data["spec_version"]
    else:
        return "2.0"

In [4]:
file_path = "../data/stix21/enterprise-attack.json"
print("Detected Version:", detect_stix_version(file_path))

Detected Version: 2.0


In [5]:
def parse_stix1(file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()

    ns = {
        "stix": "http://docs.oasis-open.org/cti/ns/stix/core-1",
        "indicator": "http://docs.oasis-open.org/cti/ns/stix/indicator-1",
        "URIObject": "http://docs.oasis-open.org/cti/ns/cybox/objects/uri-2"
    }

    records = []

    for indicator in root.findall(".//stix:Indicator", ns):
        desc_elem = indicator.find(".//indicator:Description", ns)
        value_elem = indicator.find(".//URIObject:Value", ns)

        description = desc_elem.text if desc_elem is not None else ""
        values = value_elem.text.split("##comma##") if value_elem is not None else []

        record = {
            "stix_version": "1.x",
            "entity_type": "indicator",
            "indicator_type": "url-watchlist",
            "description": description,
            "timestamp": indicator.get("timestamp"),
            "values": ",".join(values),
            "validation_flag": 1
        }

        records.append(record)

    return pd.DataFrame(records)

In [6]:
def parse_stix2(file_path):
    with open(file_path, "r") as f:
        data = json.load(f)

    records = []

    for obj in data.get("objects", []):
        record = {
            "stix_version": data.get("spec_version", "2.0"),
            "entity_type": obj.get("type"),
            "indicator_type": obj.get("name", obj.get("type")),
            "description": obj.get("description", ""),
            "timestamp": obj.get("created"),
            "values": "",
            "validation_flag": 1
        }

        refs = obj.get("external_references", [])
        values = []
        for r in refs:
            if "url" in r:
                values.append(r["url"])
            if "external_id" in r:
                values.append(r["external_id"])

        record["values"] = ",".join(values)
        records.append(record)

    return pd.DataFrame(records)

In [7]:
stix1_df = parse_stix1("../data/stix1/STIX_URL_Watchlist.xml")
stix20_df = parse_stix2("../data/stix20/enterprise-attack.json")
stix21_df = parse_stix2("../data/stix21/enterprise-attack.json")

merged_df = pd.concat([stix1_df, stix20_df, stix21_df], ignore_index=True)
merged_df = merged_df.dropna()

print("Total records:", len(merged_df))

Total records: 49544


In [8]:
def threat_label(row):
    keywords = ["malware", "attack", "phish", "ransom", "trojan"]
    text = (str(row["indicator_type"]) + str(row["description"])).lower()
    
    for word in keywords:
        if word in text:
            return 1
    return 0

merged_df["Threat_Label"] = merged_df.apply(threat_label, axis=1)

In [9]:
def credibility_score(row):
    if row["validation_flag"] != 1:
        return 0
    
    if len(str(row["values"]).split(",")) > 1:
        return 1
    
    return 0

merged_df["Credibility_Score"] = merged_df.apply(credibility_score, axis=1)

In [10]:
merged_df["ml_text"] = (
    merged_df["indicator_type"].astype(str) + " " +
    merged_df["description"].astype(str) + " " +
    merged_df["values"].astype(str)
)

vectorizer = TfidfVectorizer(stop_words="english")
X = vectorizer.fit_transform(merged_df["ml_text"])
y = merged_df["Threat_Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

merged_df["ML_Probability"] = model.predict_proba(X)[:, 1]
merged_df["ML_Prediction"] = model.predict(X)

In [11]:
with open("../data/stix21/enterprise-attack.json", "r") as f:
    stix_graph_data = json.load(f)

G = nx.DiGraph()

for obj in stix_graph_data["objects"]:
    if obj["type"] != "relationship":
        G.add_node(obj["id"])

for obj in stix_graph_data["objects"]:
    if obj["type"] == "relationship":
        G.add_edge(obj["source_ref"], obj["target_ref"])

print("Graph Nodes:", G.number_of_nodes())
print("Graph Edges:", G.number_of_edges())

Graph Nodes: 4724
Graph Edges: 20048


In [12]:
degree = nx.degree_centrality(G)
betweenness = nx.betweenness_centrality(G)
pagerank = nx.pagerank(G)

graph_risk = {}

for node in G.nodes():
    d = degree.get(node, 0)
    b = betweenness.get(node, 0)
    p = pagerank.get(node, 0)

    graph_risk[node] = 0.4*d + 0.3*b + 0.3*p

In [13]:
merged_df["Graph_Risk"] = 0

if "id" in merged_df.columns:
    merged_df["Graph_Risk"] = merged_df["id"].map(graph_risk).fillna(0)

In [14]:
merged_df["Final_Score"] = (
    0.5 * merged_df["ML_Probability"] +
    0.3 * merged_df["Graph_Risk"] +
    0.2 * merged_df["Credibility_Score"]
)

def risk_level(score):
    if score > 0.7:
        return "High"
    elif score > 0.4:
        return "Medium"
    else:
        return "Low"

merged_df["Risk_Level"] = merged_df["Final_Score"].apply(risk_level)

merged_df[["entity_type", "Final_Score", "Risk_Level"]].head()

,entity_type,Final_Score,Risk_Level
0,indicator,0.278773,Low
1,x-mitre-matrix,0.402112,Medium
2,course-of-action,0.332472,Low
3,course-of-action,0.338631,Low
4,course-of-action,0.301751,Low


In [15]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("CLASSIFICATION REPORT")
print(classification_report(y_test, y_pred))

print("CONFUSION MATRIX")
print(confusion_matrix(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.97      0.96      0.97      2579
           1       0.99      0.99      0.99      7330

    accuracy                           0.98      9909
   macro avg       0.98      0.97      0.98      9909
weighted avg       0.98      0.98      0.98      9909

CONFUSION MATRIX
[[2472  107]
 [  67 7263]]
ROC-AUC: 0.998274498560363


In [16]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(contamination=0.05, random_state=42)
merged_df["Anomaly_Score"] = iso.fit_predict(X)

merged_df["Anomaly_Flag"] = merged_df["Anomaly_Score"].apply(lambda x: 1 if x == -1 else 0)
pickle.dump(iso, open("models/iso.pkl", "wb"))

In [17]:
merged_df["Final_Risk_Score"] = (
      0.4 * merged_df["Threat_Label"]
    + 0.3 * merged_df["Credibility_Score"]
    + 0.3 * merged_df["Anomaly_Flag"]
)

In [18]:
merged_df

,stix_version,entity_type,indicator_type,description,timestamp,values,validation_flag,Threat_Label,Credibility_Score,ml_text,ML_Probability,ML_Prediction,Graph_Risk,Final_Score,Risk_Level,Anomaly_Score,Anomaly_Flag,Final_Risk_Score
0,1.x,indicator,url-watchlist,Sample URL Indicator for this watchlist,2015-05-15T09:00:00.000000Z,"http://example.com/foo/malicious1.html,http://...",1,0,1,url-watchlist Sample URL Indicator for this wa...,0.157546,0,0,0.278773,Low,1,0,0.3
1,2.0,x-mitre-matrix,Enterprise ATT&CK,Below are the tactics and technique representi...,2018-10-17T00:14:20.652Z,"https://attack.mitre.org/matrices/enterprise,e...",1,0,1,Enterprise ATT&CK Below are the tactics and te...,0.404223,0,0,0.402112,Medium,1,0,0.3
2,2.0,course-of-action,Password Filter DLL Mitigation,Ensure only valid password filters are registe...,2018-10-17T00:14:20.652Z,"https://attack.mitre.org/mitigations/T1174,T11...",1,0,1,Password Filter DLL Mitigation Ensure only val...,0.264944,0,0,0.332472,Low,1,0,0.3
3,2.0,course-of-action,Space after Filename Mitigation,Prevent files from having a trailing space aft...,2018-10-17T00:14:20.652Z,"https://attack.mitre.org/mitigations/T1151,T1151",1,0,1,Space after Filename Mitigation Prevent files ...,0.277263,0,0,0.338631,Low,1,0,0.3
4,2.0,course-of-action,HISTCONTROL Mitigation,Prevent users from changing the <code>HISTCONT...,2018-10-17T00:14:20.652Z,"https://attack.mitre.org/mitigations/T1148,T11...",1,0,1,HISTCONTROL Mitigation Prevent users from chan...,0.203501,0,0,0.301751,Low,1,0,0.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49539,2.0,relationship,relationship,[ToddyCat](https://attack.mitre.org/groups/G10...,2024-01-23T20:36:19.677Z,https://securelist.com/toddycat-keep-calm-and-...,1,1,0,relationship [ToddyCat](https://attack.mitre.o...,0.935526,1,0,0.467763,Medium,1,0,0.4
49540,2.0,relationship,relationship,[Bisonal](https://attack.mitre.org/software/S...,2022-01-27T18:04:46.484Z,https://blog.talosintelligence.com/2020/03/bis...,1,1,0,relationship [Bisonal](https://attack.mitre.o...,0.964789,1,0,0.482394,Medium,1,0,0.4
49541,2.0,relationship,relationship,[ZxShell](https://attack.mitre.org/software/S0...,2019-09-24T14:19:05.322Z,https://blogs.cisco.com/security/talos/opening...,1,1,0,relationship [ZxShell](https://attack.mitre.or...,0.982700,1,0,0.491350,Medium,1,0,0.4
49542,2.0,identity,The MITRE Corporation,,2017-06-01T00:00:00.000Z,,1,0,0,The MITRE Corporation,0.320733,0,0,0.160367,Low,1,0,0.0
